In [5]:
import json
import orjson
import glob
from pathlib import Path, WindowsPath

import pandas as pd
import numpy as np

import re

# User arguments

pattern = r'mask_tf(\d+)_apoc_cell(\d+)\.json'
port = 5022
input_path = r"H:\PROJECTS-03\Pablo\Testing\test_zyla_segmentation\segmentation_test_second\metadata\ppp002"
nan_threshold = 0.8
var_cri = 100
current_socket = 'localhost:8890'
channel = 1
df = pd.DataFrame()

# Ordering functions

def ordering_function_tf(path):
    """Give the timeframe contained in the name of the json file as an integer.
    Use to sort metadata files"""
    
    pattern = r'mask_tf(\d+)_apoc_cell(\d+)\.json'
    match = re.match(pattern, path.name)
    return int(match.group(1))

def ordering_function_cell(path):
    """Give the cell number contained in the name of the json file as an integer.
    Use to sort metadata files"""

    pattern = r'mask_tf(\d+)_apoc_cell(\d+)\.json'
    match = re.match(pattern, path.name)
    return int(match.group(2))

def process_subdirectory(subdirectory_path, pattern=pattern, df=df, channel=channel):
    json_files = list(subdirectory_path.glob('mask_tf*_apoc_cell*.json'))
    json_files.sort(key=lambda x: (ordering_function_cell(x), ordering_function_tf(x)))
    cells = {re.match(pattern, file.name).group(2) for file in json_files} # Cell numbers as in the metadata file names

    # Check whether cells is emptyy or metadata has been found, if so populate df with average intenstiy
    if cells:
        for cell in cells:
            print(f'Recording cells {subdirectory.name}+{cell}')
            json_cell = [json_file for json_file in json_files if 'apoc_cell'+cell in json_file.stem]
            tfs = [int((re.match(pattern,json_file.name)).group(1)) for json_file in json_cell]
            tfs.sort()
            for (tf, json_file) in zip(tfs, json_cell):
                with json_file.open('rb') as file:
                    data_dict = orjson.loads(file.read())
                    npixels = data_dict.get('npixels')
                    intensity = data_dict.get('intensity')[channel] if 'intensity' in data_dict and len(data_dict['intensity']) > 1 else None
                    column_name = subdirectory_path.name+'_'+cell
                    df.at[tf, column_name] = intensity/npixels
    else:
        print(f'No cell metadata in {subdirectory_path.name}')
    
    return df
    
    # except AttributeError as e:
    #     print(f'No cell metadata in {subdirectory_path.name}: {e}')
    #     pass

In [6]:
# %%timeit -n5 -r3
root_folder = Path(input_path)
subdirectories = list(root_folder.glob('*/'))

In [7]:
subdirectories

[WindowsPath('H:/PROJECTS-03/Pablo/Testing/test_zyla_segmentation/segmentation_test_second/metadata/ppp002/ppp002_xy01'),
 WindowsPath('H:/PROJECTS-03/Pablo/Testing/test_zyla_segmentation/segmentation_test_second/metadata/ppp002/ppp002_xy02'),
 WindowsPath('H:/PROJECTS-03/Pablo/Testing/test_zyla_segmentation/segmentation_test_second/metadata/ppp002/ppp002_xy03'),
 WindowsPath('H:/PROJECTS-03/Pablo/Testing/test_zyla_segmentation/segmentation_test_second/metadata/ppp002/ppp002_xy04'),
 WindowsPath('H:/PROJECTS-03/Pablo/Testing/test_zyla_segmentation/segmentation_test_second/metadata/ppp002/ppp002_xy05'),
 WindowsPath('H:/PROJECTS-03/Pablo/Testing/test_zyla_segmentation/segmentation_test_second/metadata/ppp002/ppp002_xy06'),
 WindowsPath('H:/PROJECTS-03/Pablo/Testing/test_zyla_segmentation/segmentation_test_second/metadata/ppp002/ppp002_xy07'),
 WindowsPath('H:/PROJECTS-03/Pablo/Testing/test_zyla_segmentation/segmentation_test_second/metadata/ppp002/ppp002_xy08'),
 WindowsPath('H:/PROJECT

In [8]:
for subdirectory in subdirectories:
    process_subdirectory(subdirectory, df=df)

# Filter traces with nan_values defined by nan_threshold

Recording cells ppp002_xy01+0
Recording cells ppp002_xy02+0
Recording cells ppp002_xy03+2
Recording cells ppp002_xy03+0
Recording cells ppp002_xy03+1
Recording cells ppp002_xy03+3
Recording cells ppp002_xy04+0
Recording cells ppp002_xy05+0
Recording cells ppp002_xy06+2
Recording cells ppp002_xy06+0
Recording cells ppp002_xy06+1
Recording cells ppp002_xy07+0
Recording cells ppp002_xy08+0
Recording cells ppp002_xy08+1
Recording cells ppp002_xy09+0
Recording cells ppp002_xy09+1
Recording cells ppp002_xy10+0
Recording cells ppp002_xy11+0
Recording cells ppp002_xy12+0
Recording cells ppp002_xy13+0
Recording cells ppp002_xy14+0
Recording cells ppp002_xy15+0
Recording cells ppp002_xy16+0
Recording cells ppp002_xy17+0
Recording cells ppp002_xy18+0
Recording cells ppp002_xy19+0
Recording cells ppp002_xy19+1
Recording cells ppp002_xy20+2
Recording cells ppp002_xy20+0
Recording cells ppp002_xy20+1
Recording cells ppp002_xy21+0
Recording cells ppp002_xy22+0
Recording cells ppp002_xy23+0
Recording 

C:\Users\pperez\AppData\Local\Temp\1\ipykernel_38188\2418443906.py:58: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.at[tf, column_name] = intensity/npixels


Recording cells ppp002_xy73+2
Recording cells ppp002_xy73+0


C:\Users\pperez\AppData\Local\Temp\1\ipykernel_38188\2418443906.py:58: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.at[tf, column_name] = intensity/npixels
C:\Users\pperez\AppData\Local\Temp\1\ipykernel_38188\2418443906.py:58: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.at[tf, column_name] = intensity/npixels


Recording cells ppp002_xy73+1
Recording cells ppp002_xy74+2
Recording cells ppp002_xy74+0


...

Recording cells ppp002_xy82+1


C:\Users\pperez\AppData\Local\Temp\1\ipykernel_38188\2418443906.py:58: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.at[tf, column_name] = intensity/npixels


Recording cells ppp002_xy83+0


C:\Users\pperez\AppData\Local\Temp\1\ipykernel_38188\2418443906.py:58: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.at[tf, column_name] = intensity/npixels


No cell metadata in project.json


In [3]:
df_cleaned = df.drop(columns=df.columns[df.isna().sum() > nan_threshold*len(df)])

# Filter values with variance criterion defined in var_cri

variances = df_cleaned.var()
df_cleaned = df_cleaned.loc[:, variances > var_cri]

In [8]:
port=5024

In [9]:
from bokeh.io import output_notebook, show, push_notebook
from bokeh.layouts import column, row
from bokeh.models import Button, ColumnDataSource, CustomJS
from bokeh.plotting import figure, curdoc
from bokeh.application.handlers import FunctionHandler
from bokeh.application import Application
from bokeh.server.server import Server
from tornado.ioloop import IOLoop
import asyncio


global valid_signals

def modify_doc(doc):
    # Initialize selected_plots as a ColumnDataSource
    selected_plots_source = ColumnDataSource(data=dict(selected_plots=[]))

    # Function to get selected plots in Python
    def get_selected_plots():
        return selected_plots_source.data['selected_plots']

    # Function to create plots and buttons layout
    def create_plots_layout():
        plots = []
        buttons = []

        for col in df_cleaned.columns:
            if col not in get_selected_plots():
                source = ColumnDataSource(data={col: df_cleaned[col], 'x': range(len(df_cleaned))})
                p = figure(width=250, height=250, title=col)
                r = p.line('x', col, source=source, line_width=2, color='navy', alpha=0.8)

                button = Button(label=col, width=60, button_type="success")

                def create_button_callback(plot, column_name, btn):
                    def callback():
                        selected_plots = selected_plots_source.data['selected_plots']
                        if column_name in selected_plots:
                            plot.background_fill_color = 'white'
                            selected_plots.remove(column_name)
                            btn.button_type = 'success'
                        else:
                            plot.background_fill_color = 'rgba(255, 0, 0, 0.1)'
                            selected_plots.append(column_name)
                            btn.button_type = 'danger'
                        selected_plots_source.data = {'selected_plots': selected_plots}  # Update the data source
                        global valid_signals
                        valid_signals = df_cleaned.drop(columns=selected_plots_source.data['selected_plots']).columns
                        # push_notebook()  # Ensure updates are reflected in the notebook
                    return callback

                button.on_click(create_button_callback(p, col, button))

                plots.append(p)
                buttons.append(button)

        # Organize layout
        plot_rows = []
        for i in range(0, len(plots), 5):
            plot_row = plots[i:i+5]
            button_row = buttons[i:i+5]
            plot_rows.append(row(*plot_row, column(*button_row)))

        layout = column(*plot_rows)
        return layout

    # Create the initial layout
    layout = create_plots_layout()

    # Button to print excluded plots
    print_button = Button(label="Print list of excluded plots", width=200, button_type="primary")
    def print_selected_plots():
        print(get_selected_plots())
        # push_notebook()  # Ensure notebook updates
    print_button.on_click(print_selected_plots)

    # Button to rerender without selected plots
    rerender_button = Button(label="Exclude selected plots", width=200, button_type="warning")
    def rerender_plots():
        new_layout = create_plots_layout()
        doc.clear()  # Clear the current document
        doc.add_root(column(new_layout, print_button, rerender_button))
        # push_notebook()
    rerender_button.on_click(rerender_plots)

    # Add the layout and buttons to the current document
    doc.add_root(column(layout, print_button, rerender_button))

# Create the application
app = Application(FunctionHandler(modify_doc))


# Display app
# show(app, notebook_handle=True) - Carefull to push notebook

# Or do it through a server

#  Start the Bokeh server
# server = Server({'/': modify_doc}, port=4996)
#  server.start()

# def show_app():
#     server.io_loop.add_callback(server.show, "/")
#     server.io_loop.start()

# show_app()

#########################
# Integrate with the current Jupyter server -- To be checked
server = Server({'/': modify_doc}, port=port, io_loop=IOLoop.current(), allow_websocket_origin=[current_socket, "localhost:"+str(port)])

async def show_app():
    server.io_loop.add_callback(server.show, "/")
    await server.io_loop.start()

# Integrate with the Jupyter notebook event loop
loop = asyncio.get_event_loop()
if loop.is_running():
    loop.create_task(show_app())
else:
    loop.run_until_complete(show_app())

# Store valid signals

Task exception was never retrieved
future: <Task finished name='Task-338' coro=<show_app() done, defined at C:\Users\pperez\AppData\Local\Temp\1\ipykernel_24904\2203314975.py:112> exception=RuntimeError('This event loop is already running')>
Traceback (most recent call last):
  File "C:\Users\pperez\AppData\Local\Temp\1\ipykernel_24904\2203314975.py", line 114, in show_app
    await server.io_loop.start()
  File "C:\Users\pperez\AppData\Roaming\Python\Python39\site-packages\tornado\platform\asyncio.py", line 205, in start
    self.asyncio_loop.run_forever()
  File "C:\Users\pperez\.conda\envs\devbio-napari-cupy\lib\asyncio\base_events.py", line 591, in run_forever
    self._check_running()
  File "C:\Users\pperez\.conda\envs\devbio-napari-cupy\lib\asyncio\base_events.py", line 583, in _check_running
    raise RuntimeError('This event loop is already running')
RuntimeError: This event loop is already running


In [11]:
df_cleaned = df[valid_signals]

In [12]:
df_interpolate = df_cleaned.interpolate(method='linear', axis=1)

In [13]:
df_interpolate

,ppp003_xy13_1,ppp003_xy13_0,ppp003_xy16_0,ppp003_xy17_0,ppp003_xy18_1,ppp003_xy18_0,ppp003_xy19_0,ppp003_xy20_1,ppp003_xy20_0,ppp003_xy21_0,ppp003_xy22_0,ppp003_xy23_0
6,64.480902,25.139806,113.463811,100.609275,142.481251,175.909574,20.328319,117.853650,115.725234,95.295209,53.773752,27.290057
7,66.060367,32.437989,71.913524,70.261942,129.365725,136.116865,52.618942,115.423091,86.382188,91.916309,54.749218,24.639596
0,78.902296,80.382118,174.510979,32.795741,97.876013,246.895275,19.036478,101.591822,51.546989,94.893632,66.667754,67.668308
1,84.497863,42.974587,126.869260,24.926465,80.353435,168.768586,24.497253,94.765692,62.218379,97.495563,55.036178,71.794033
2,75.219692,33.306241,112.604626,24.224707,93.672963,135.809182,33.754131,106.541997,125.992212,99.970795,46.924106,60.283172
...,...,...,...,...,...,...,...,...,...,...,...,...
116,29.862273,22.105813,680.156820,74.127102,77.888763,57.846330,37.803897,22.710862,72.311843,22.230432,21.846186,21.846186
117,31.720191,22.090616,611.087846,78.454365,85.565124,61.094945,36.624766,24.395949,67.712707,22.529610,22.056617,22.056617
118,26.703745,20.572548,632.537508,79.916618,72.739426,54.831659,36.923892,22.190196,72.467661,22.160561,21.722232,21.722232
119,29.605588,20.190767,574.304575,75.569184,79.635896,59.348138,39.060379,22.822163,75.762582,22.295653,22.063698,22.063698


In [14]:
df_interpolate.to_csv('ppp003_test')

In [11]:
import pandas as pd
import numpy as np
from bokeh.plotting import figure, show, output_notebook
from bokeh.layouts import gridplot
from bokeh.models import ColumnDataSource, HoverTool

# Enable Bokeh output in the notebook
output_notebook()

# Number of plots per row
plots_per_row = 5

# Create a list to hold all the plots
plots = []

for col in df_cleaned.columns:
    # Create a ColumnDataSource for each column
    source = ColumnDataSource(data=dict(index=df_cleaned.index, values=df_interpolate[col]))

    # Create a new plot
    p = figure(title=col, plot_width=275, plot_height=275, tools="hover", tooltips="@index: @values")

    # Add a line renderer with legend and line thickness
    p.line('index', 'values', source=source, line_width=2)
    
    # Add the plot to the list of plots
    plots.append(p)

# Arrange the plots in a grid
grid = gridplot(plots, ncols=plots_per_row)

# Show the results in the notebook
show(grid)

Loading BokehJS ...

In [3]:
server.io_loop.stop()